# Cobalt L-edge energy sweep: universal multi-energy phase retrieval

Jointly reconstruct the fixed magnetic state from ideal CR (`+1`) and CL (`-1`) holograms across the manifest-listed energy sweep. The loading, optional detector-energy rescaling, crop/bin controls, support preparation, and exploratory same-energy CR/CL retrieval mirror `03_phase_retrieval_core_multienergy.ipynb`, while the final retrieval remains the universal multi-polarization model. The stored `xmcd_log/data` arrays are loaded only at the end for validation.


In [ ]:
from pathlib import Path
import json
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage as ndi

# Make imports work when Jupyter starts in either the repository root or notebooks/.
REPO_ROOT = Path.cwd() if (Path.cwd() / "library").is_dir() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from library import phase_retrieval_universal as pr
from library import interactive


REPO_ROOT =Path.cwd().parent.parent


In [ ]:
plt.rcParams.update({"figure.figsize": (7, 5), "image.cmap": "gray"})


## Load the cobalt energy chunks

Each chunk contains ideal CR and CL holograms, a beamstop mask, a support mask, and effective refractive-index arrays. Both polarizations are loaded and later interleaved for the universal reconstruction.


In [ ]:
DATA_DIR = REPO_ROOT / "Data" / "prop_False_proj_False"
MANIFEST_FILE = DATA_DIR / "manifest.json"


load_hologram = "ideal_holograms"

if not MANIFEST_FILE.exists():
    raise FileNotFoundError(MANIFEST_FILE)

with MANIFEST_FILE.open("r") as handle:
    manifest = json.load(handle)[::2]

chunk_entries = sorted(manifest, key=lambda item: item["index"])
chunk_files = [DATA_DIR / entry["file"] for entry in chunk_entries]
missing_files = [path for path in chunk_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing energy chunk files: {missing_files}")

energies_eV = np.asarray([entry["energy_eV"] for entry in chunk_entries], dtype=float)
group_names = [path.stem for path in chunk_files]

cr_ideal = []
cl_ideal = []
beamstop_masks = []
supportmasks = []
effective_refractive_indices_m = []
for chunk_index, (expected_energy, chunk_file) in enumerate(zip(energies_eV, chunk_files)):
    with h5py.File(chunk_file, "r") as handle:
        file_energy = float(handle.attrs["energy_eV"])
        if not np.isclose(file_energy, expected_energy):
            raise ValueError(
                f"Manifest energy {expected_energy} eV does not match "
                f"{chunk_file.name} energy {file_energy} eV"
            )

        cr_ideal.append(np.squeeze(np.asarray(handle[f"{load_hologram}/CR"], dtype=float)))
        cl_ideal.append(np.squeeze(np.asarray(handle[f"{load_hologram}/CL"], dtype=float)))
        beamstop_masks.append(np.asarray(handle["beamstop_mask/data"], dtype=float))
        supportmasks.append(np.asarray(handle["supportmask/data"], dtype=float))
        effective_refractive_indices_m.append(
            np.asarray(handle["material_layers/effective_refractive_indices_m"])
        )

        if chunk_index == 0:
            layer_names = np.asarray(handle["material_layers/layer_names"]).astype(str)
            refractive_index_channel_names = np.asarray(
                handle["material_layers/refractive_index_channel_names"]
            ).astype(str)

cr_ideal = np.stack(cr_ideal)
cl_ideal = np.stack(cl_ideal)
beamstop_masks = np.stack(beamstop_masks)
supportmasks = np.stack(supportmasks)
effective_refractive_indices_m = np.stack(effective_refractive_indices_m)
emin_index = int(np.argmin(energies_eV))
emin_group = group_names[emin_index]
supportmask = supportmasks[emin_index]

if cr_ideal.shape != cl_ideal.shape:
    raise ValueError(f"CR and CL shapes differ: {cr_ideal.shape} vs {cl_ideal.shape}")
if cr_ideal.shape[0] != len(energies_eV):
    raise ValueError("The hologram energy axis does not match energies")
if np.any(~np.isfinite(cr_ideal)) or np.any(~np.isfinite(cl_ideal)):
    raise ValueError("The ideal holograms contain NaN or infinite values")
if np.min(cr_ideal) < 0 or np.min(cl_ideal) < 0:
    raise ValueError("The ideal holograms must be non-negative intensities")

n_energy, ny, nx = cr_ideal.shape
print(f"energies: {energies_eV[0]:.1f} to {energies_eV[-1]:.1f} eV ({n_energy} points)")
print("CR stack:", cr_ideal.shape, cr_ideal.dtype)
print("CL stack:", cl_ideal.shape, cl_ideal.dtype)
print("effective refractive indices:", effective_refractive_indices_m.shape, effective_refractive_indices_m.dtype)
print("refractive-index channels:", list(refractive_index_channel_names))
print(f"Emin support: {emin_group} at {energies_eV[emin_index]:.1f} eV")


## Plot effective refractive indices

The HDF5 chunks store `material_layers/effective_refractive_indices_m` with shape `(n_energy, n_layer, n_channel)`. Select a layer below to inspect its complex effective refractive-index channels as a function of energy.

In [ ]:
%matplotlib widget
plt.close("all")

REFRACTIVE_INDEX_LAYER = -1  # int index or layer name string

if isinstance(REFRACTIVE_INDEX_LAYER, str):
    matches = np.flatnonzero(layer_names == REFRACTIVE_INDEX_LAYER)
    if matches.size == 0:
        raise ValueError(f"Unknown layer name: {REFRACTIVE_INDEX_LAYER!r}")
    layer_index = int(matches[0])
else:
    layer_index = int(REFRACTIVE_INDEX_LAYER) % len(layer_names)

layer_n_eff = effective_refractive_indices_m[:, layer_index, :]
layer_label = layer_names[layer_index]

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
for channel_index, channel_name in enumerate(refractive_index_channel_names):
    values = layer_n_eff[:, channel_index]
    axes[0].plot(energies_eV, np.real(values-values[0]), "o-", label=channel_name)
    axes[1].plot(energies_eV, np.imag(values-values[0]), "o-", label=channel_name)
axes[0].set_ylabel("real(n_eff)")
axes[1].set_ylabel("imag(n_eff)")
axes[1].set_xlabel("Energy (eV)")
axes[0].set_title(f"Effective refractive indices: layer {layer_index} ({layer_label})")
for axis in axes:
    axis.grid(True, alpha=0.3)
    axis.legend()
plt.tight_layout()


In [ ]:
%matplotlib widget

show_indices = np.unique(np.linspace(0, n_energy - 1, 6, dtype=int))
fig, axes = plt.subplots(2, len(show_indices), figsize=(len(show_indices)*3, 6), squeeze=False, sharex=True, sharey=True)
for column, index in enumerate(show_indices):
    for row, (stack, label) in enumerate(((cr_ideal, "CR"), (cl_ideal, "CL"))):
        axes[row, column].imshow(np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(stack[index])))))
        axes[row, column].set_title(f"{label}, {energies_eV[index]:.1f} eV")
        axes[row, column].axis("off")
plt.suptitle("Ideal hologram intensities, log display")
plt.tight_layout()

In [ ]:
%matplotlib widget

show_indices = np.unique(np.linspace(0, n_energy - 1, 3, dtype=int))
fig, axes = plt.subplots(2, len(show_indices), figsize=(len(show_indices)*3, 6), squeeze=False, sharex=True, sharey=True)
for column, index in enumerate(show_indices):
    for row, (stack, label) in enumerate(((cr_ideal, "CR"), (cl_ideal, "CL"))):
        axes[row, column].imshow(np.log1p(stack[index]))
        axes[row, column].set_title(f"{label}, {energies_eV[index]:.1f} eV")
        axes[row, column].axis("off")
plt.suptitle("Ideal hologram intensities, log display")
plt.tight_layout()

## Normalize detector sampling and build metadata

The reciprocal-space scale changes with photon energy. When `rescale=True`, each hologram is stretched about its center by `E/Emin` and sampled back onto the original grid. Set `rescale=False` to keep the loaded detector arrays unchanged while still using the same crop/bin preparation.

After that, the optional `PRE_RETRIEVAL_CROP_SHAPE` and `PRE_RETRIEVAL_BIN_FACTOR` controls prepare smaller arrays for phase retrieval. Holograms and `mask_pixel` are cropped and binned in detector space in the same way. The support template is treated differently: detector cropping rescales the support to the new array shape so it occupies the same fraction of the image, while detector binning center-crops the support to the binned shape.


In [ ]:
def centered_rescale(image, scale, output_shape=None, order=1, rescale=True):
    """Rescale about the array center and return a fixed-size center crop."""
    image = np.asarray(image)
    if image.ndim != 2:
        raise ValueError(f"Expected a 2D image, got shape {image.shape}")
    if scale <= 0 or not np.isfinite(scale):
        raise ValueError("scale must be positive and finite")
    if output_shape is None:
        output_shape = image.shape
    output_shape = tuple(int(value) for value in output_shape)
    if len(output_shape) != 2 or min(output_shape) <= 0:
        raise ValueError("output_shape must contain two positive integers")
    if np.isclose(scale, 1.0) and output_shape == image.shape:
        return image.copy()

    input_center = (np.asarray(image.shape, dtype=float) - 1.0) / 2.0
    output_center = (np.asarray(output_shape, dtype=float) - 1.0) / 2.0
    output_indices = np.indices(output_shape, dtype=float)
    coordinates = (
        (output_indices.reshape(2, -1) - output_center[:, None]) / float(scale)
        + input_center[:, None]
    )
    if rescale:
        return ndi.map_coordinates(
            image,
            coordinates,
            order=order,
            mode="constant",
            cval=0.0,
            prefilter=False,
        ).reshape(output_shape)
    return image.copy()


def centered_resize(image, output_shape, order=1):
    """Resize a 2D image to output_shape while preserving relative position."""
    image = np.asarray(image)
    output_shape = tuple(int(value) for value in output_shape)
    if image.ndim != 2:
        raise ValueError(f"Expected a 2D image, got shape {image.shape}")
    if len(output_shape) != 2 or min(output_shape) <= 0:
        raise ValueError("output_shape must contain two positive integers")
    if image.shape == output_shape:
        return image.copy()

    input_center = (np.asarray(image.shape, dtype=float) - 1.0) / 2.0
    output_center = (np.asarray(output_shape, dtype=float) - 1.0) / 2.0
    matrix = np.diag(np.asarray(image.shape, dtype=float) / np.asarray(output_shape, dtype=float))
    offset = input_center - matrix @ output_center
    return ndi.affine_transform(
        image,
        matrix=matrix,
        offset=offset,
        output_shape=output_shape,
        order=order,
        mode="constant",
        cval=0.0,
        prefilter=False,
    )


def normalized_crop_shape(crop_shape, image_shape):
    """Return a validated crop shape or None when cropping is disabled."""
    if crop_shape is None:
        return None
    if isinstance(crop_shape, (int, np.integer)):
        crop_shape = (int(crop_shape), int(crop_shape))
    crop_shape = tuple(int(value) for value in crop_shape)
    if len(crop_shape) != 2 or min(crop_shape) <= 0:
        raise ValueError("PRE_RETRIEVAL_CROP_SHAPE must be None, an int, or (ny, nx)")
    if crop_shape[0] > image_shape[0] or crop_shape[1] > image_shape[1]:
        raise ValueError("PRE_RETRIEVAL_CROP_SHAPE cannot exceed the hologram shape")
    return crop_shape


def center_crop_2d(image, output_shape):
    """Center-crop a 2D image to output_shape."""
    image = np.asarray(image)
    output_shape = tuple(int(value) for value in output_shape)
    if image.ndim != 2:
        raise ValueError(f"Expected a 2D image, got shape {image.shape}")
    if output_shape[0] > image.shape[0] or output_shape[1] > image.shape[1]:
        raise ValueError("Cannot center-crop to a larger shape")
    start_y = (image.shape[0] - output_shape[0]) // 2
    start_x = (image.shape[1] - output_shape[1]) // 2
    return image[start_y:start_y + output_shape[0], start_x:start_x + output_shape[1]].copy()


def center_crop_stack(stack, output_shape):
    """Center-crop the last two axes of a stack."""
    stack = np.asarray(stack)
    output_shape = tuple(int(value) for value in output_shape)
    if output_shape[0] > stack.shape[-2] or output_shape[1] > stack.shape[-1]:
        raise ValueError("Cannot center-crop to a larger shape")
    start_y = (stack.shape[-2] - output_shape[0]) // 2
    start_x = (stack.shape[-1] - output_shape[1]) // 2
    return stack[..., start_y:start_y + output_shape[0], start_x:start_x + output_shape[1]].copy()


def bin_stack(stack, factor, reducer="mean"):
    """Bin the last two axes of a stack after cropping to a multiple of factor."""
    factor = int(factor)
    if factor <= 0:
        raise ValueError("PRE_RETRIEVAL_BIN_FACTOR must be a positive integer")
    stack = np.asarray(stack)
    if factor == 1:
        return stack.copy()

    binned_shape = (stack.shape[-2] // factor, stack.shape[-1] // factor)
    if min(binned_shape) <= 0:
        raise ValueError("PRE_RETRIEVAL_BIN_FACTOR is too large for the image shape")
    crop_shape = (binned_shape[0] * factor, binned_shape[1] * factor)
    cropped = center_crop_stack(stack, crop_shape)
    reshaped = cropped.reshape(*cropped.shape[:-2], binned_shape[0], factor, binned_shape[1], factor)
    if reducer == "mean":
        return reshaped.mean(axis=(-3, -1))
    if reducer == "max":
        return reshaped.max(axis=(-3, -1))
    raise ValueError("reducer must be 'mean' or 'max'")


def build_support_template(support):
    """Build the shifted support template used for phase retrieval."""
    from skimage.draw import disk

    support = np.asarray(support) != 0
    structure = np.zeros((15, 15), dtype=bool)
    yy, xx = disk((structure.shape[0] // 2, structure.shape[1] // 2), 5)
    structure[yy, xx] = True

    shift_x = -(167 - support.shape[1] // 2)
    shift_y = -(345 - support.shape[0] // 2)
    rolled_dilated = np.roll(
        np.roll(ndi.binary_dilation(support, structure=structure), shift=shift_x, axis=1),
        shift=shift_y,
        axis=0,
    )
    rolled_support = np.roll(
        np.roll(support, shift=shift_x, axis=1),
        shift=shift_y,
        axis=0,
    )
    template = rolled_dilated.astype(bool)
    x_cut = min(300, template.shape[1])
    rolled_support=ndi.binary_erosion(rolled_support,iterations=3)
    template[:, :x_cut] = rolled_support[:, :x_cut]

    template=np.roll(np.roll(template, shift=-shift_x, axis=1), shift=-shift_y, axis=0)
   
    return template


def preprocess_detector_stack(stack, crop_shape, bin_factor, reducer="mean"):
    """Apply the detector-space crop/bin operations to holograms or masks."""
    prepared = np.asarray(stack).copy()
    if crop_shape is not None:
        prepared = center_crop_stack(prepared, crop_shape)
    if bin_factor > 1:
        prepared = bin_stack(prepared, bin_factor, reducer=reducer)
    return prepared


def preprocess_support_template(template, crop_shape, bin_factor, final_shape):
    """Apply the support-specific crop/bin rules requested for this notebook."""
    prepared = np.asarray(template) != 0
    if crop_shape is not None:
        # Cropping detector data changes the array shape. For the support, keep
        # the same fractional position and size instead of cutting pixels away.
        prepared = centered_resize(prepared.astype(float), crop_shape, order=0) > 0.5
    if bin_factor > 1:
        # Detector binning reduces the Fourier grid. For the support template,
        # keep the central real-space region with the final binned shape.
        prepared = center_crop_2d(prepared, final_shape) != 0
    return prepared.astype(bool)


rescale = False
# Optional phase-retrieval preprocessing controls.
# Examples: PRE_RETRIEVAL_CROP_SHAPE = (384, 384), PRE_RETRIEVAL_BIN_FACTOR = 2.
PRE_RETRIEVAL_CROP_SHAPE = None
PRE_RETRIEVAL_BIN_FACTOR = 1
USE_MASK_PIXEL = load_hologram != "ideal_holograms"

emin_eV = float(energies_eV[emin_index])
scale_factors = energies_eV / emin_eV
cr_rescaled = np.stack([
    centered_rescale(image, scale, output_shape=(ny, nx), order=5, rescale=rescale)
    for image, scale in zip(cr_ideal, scale_factors)
])
cl_rescaled = np.stack([
    centered_rescale(image, scale, output_shape=(ny, nx), order=5, rescale=rescale)
    for image, scale in zip(cl_ideal, scale_factors)
])
mask_pixel_energy = np.stack([
    centered_rescale(image, scale, output_shape=(ny, nx), order=5, rescale=rescale)
    for image, scale in zip(beamstop_masks, scale_factors)
])

# Linear interpolation of non-negative inputs should remain non-negative;
# clip tiny floating-point undershoots defensively.
cr_rescaled = np.maximum(cr_rescaled, 0.0)
cl_rescaled = np.maximum(cl_rescaled, 0.0)
mask_pixel_energy = np.maximum(mask_pixel_energy, 0.0)

crop_shape = normalized_crop_shape(PRE_RETRIEVAL_CROP_SHAPE, (ny, nx))
bin_factor = int(PRE_RETRIEVAL_BIN_FACTOR)
supptemp_full = build_support_template(supportmask)

cr_prepared = preprocess_detector_stack(cr_rescaled, crop_shape, bin_factor, reducer="mean")
cl_prepared = preprocess_detector_stack(cl_rescaled, crop_shape, bin_factor, reducer="mean")
mask_pixel_energy = preprocess_detector_stack(mask_pixel_energy, crop_shape, bin_factor, reducer="max")
supptemp = preprocess_support_template(
    supptemp_full,
    crop_shape,
    bin_factor,
    final_shape=cr_prepared.shape[-2:],
)
ny_pr, nx_pr = cr_prepared.shape[-2:]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(supportmask)
axes[0].set_title(f"Raw support: {emin_eV:.1f} eV")
axes[1].imshow(supptemp)
axes[1].set_title("Support for retrieval")
axes[2].imshow(np.log1p(cr_ideal[-1]))
axes[2].set_title(f"Raw CR at {energies_eV[-1]:.1f} eV")
axes[3].imshow(np.log1p(cr_prepared[-1]))
axes[3].set_title("Prepared CR for retrieval")
for axis in axes:
    axis.axis("off")
plt.tight_layout()

print(f"Raw support pixels: {int(np.sum(supportmask != 0))} / {supportmask.size}")
print(f"Retrieval support pixels: {int(supptemp.sum())} / {supptemp.size}")
print(f"Scale-factor range: {scale_factors.min():.6f} to {scale_factors.max():.6f}")
print(f"Pre-retrieval crop shape: {crop_shape}")
print(f"Detector energy rescaling enabled: {rescale}")
print(f"Pre-retrieval bin factor: {bin_factor}")
print("Prepared stacks:", cr_prepared.shape, cl_prepared.shape)
print("Prepared mask_pixel:", mask_pixel_energy.shape, "use mask:", USE_MASK_PIXEL)


In [ ]:
# Interleave CR and CL so each adjacent pair belongs to the same energy.
holograms = np.stack([
    image
    for energy_index in range(n_energy)
    for image in (cr_prepared[energy_index], cl_prepared[energy_index])
])
mask_pixel = np.stack([
    image
    for energy_index in range(n_energy)
    for image in (mask_pixel_energy[energy_index], mask_pixel_energy[energy_index])
])
retrieval_mask_pixel = mask_pixel if USE_MASK_PIXEL else np.zeros_like(mask_pixel)
state_labels = ["fixed_state"] * (2 * n_energy)
energy_labels = np.repeat(energies_eV, 2)
polarizations = np.tile([+1.0, -1.0], n_energy)  # CR positive, CL negative
illumination_labels = ["fixed_beam"] * (2 * n_energy)

print("joint hologram stack:", holograms.shape)
print("joint mask stack:", retrieval_mask_pixel.shape)
print("support template:", supptemp.shape)
print("first four (energy, polarization):", list(zip(energy_labels, polarizations))[:4])

In [ ]:
%matplotlib widget
plt.close("all")

show_indices = np.unique(np.linspace(0, n_energy - 1, 3, dtype=int))
fig, axes = plt.subplots(3, len(show_indices), figsize=(len(show_indices) * 3, 7), sharex=True, sharey=True)
for column, energy_index in enumerate(show_indices):
    observation = 2 * energy_index
    label = "CR"
    axes[0, column].imshow(
         #np.roll(np.roll(supptemp, shift=-39, axis=0), shift=-39, axis=1)*
         np.fft.fftshift(np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(holograms[observation]))))),
        vmin=0,
        vmax=11,
    )
    axes[0, column].set_title(f"{label}, {energies_eV[energy_index]:.1f} eV")
    axes[0, column].axis("off")
    axes[1, column].imshow(np.log1p(holograms[observation]))
    axes[1, column].set_title("Prepared hologram")
    axes[1, column].axis("off")
    axes[2, column].imshow(retrieval_mask_pixel[observation])
    axes[2, column].set_title("Prepared mask_pixel")
    axes[2, column].axis("off")
plt.suptitle("Prepared holograms, support, and mask")
plt.tight_layout()

In [ ]:
import scipy
supptemp=np.zeros(holograms[0].shape)
from skimage.draw import disk
yy,xx=disk(tuple(np.array(supptemp.shape)/2), 12)
supptemp[yy,xx]=1
#yy,xx=disk(((415+294)/2,(415+294)/2 ), 4+(415-294)/2)
#supptemp[yy,xx]=1
yy,xx=disk(((415+294)/2,(415+294)/2 ), -1+(415-294)/2)
supptemp[yy,xx]=1
yy,xx=disk((197,512), 8)
supptemp[yy,xx]=1

#supptemp*=(1.*(np.abs(interactive.reconstruct(holograms[0]))>9.e-6))


#supptemp*=(1.*(np.abs(interactive.reconstruct(holograms[0]))>0.5e-6))
supptemp = 1.*scipy.ndimage.binary_fill_holes(supptemp.astype(int))
mmask=np.zeros(supptemp.shape)
yy,xx=disk(tuple(np.array(supptemp.shape)/2), 17)
mmask[yy,xx]=1
interactive.cimshow(supptemp*np.log10(np.abs(interactive.reconstruct(holograms[2]))))
#interactive.cimshow(mmask)

## Initial same-energy CR/CL pair reconstructions

Before the universal multi-polarization retrieval, reconstruct selected matching CR/CL pairs independently at the same photon energies. This mirrors the exploratory check in `03_phase_retrieval_core_multienergy.ipynb`, but keeps the exploratory support and aperture separate from the final universal support template.


In [ ]:
from skimage.draw import disk

PAIR_RETRIEVAL_QUICK_RUN = QUICK_RUN if "QUICK_RUN" in globals() else False
PAIR_RETRIEVAL_INDICES = show_indices  # Use np.arange(n_energy) to run every energy pair.

exploratory_support = np.zeros(cr_prepared.shape[-2:], dtype=float)
center = tuple(np.asarray(exploratory_support.shape) / 2)
yy, xx = disk(center, 8, shape=exploratory_support.shape)
exploratory_support[yy, xx] = 1.0

# Shape-scaled version of the larger off-center support disk used in notebook 03.
scale_y = exploratory_support.shape[0] / 512
scale_x = exploratory_support.shape[1] / 512
large_center = ((415 + 294) / 2 * scale_y, (415 + 294) / 2 * scale_x)
large_radius = (2 + (415 - 294) / 2) * min(scale_y, scale_x)
yy, xx = disk(large_center, large_radius, shape=exploratory_support.shape)
exploratory_support[yy, xx] = 1.0

reference_center = (197 * scale_y, 512 * scale_x)
yy, xx = disk(reference_center, 9 * min(scale_y, scale_x), shape=exploratory_support.shape)
exploratory_support[yy, xx] = 1.0
exploratory_support = ndi.binary_fill_holes(exploratory_support.astype(bool)).astype(float)

pair_aperture = np.zeros_like(exploratory_support)
yy, xx = disk(center, 30 * min(scale_y, scale_x), shape=pair_aperture.shape)
pair_aperture[yy, xx] = 1.0


if PAIR_RETRIEVAL_QUICK_RUN:
    pair_number_iterations = [5, 1, 5, 1]
    pair_average_img = [1, 1, 1, 1]
    pair_plot_every = [1e9, 1e9, 1e9, 1e9]
else:
    pair_number_iterations = [30, 18, 13, 13, 10000, 33]
    pair_average_img = [10]*len(pair_number_iterations)
    pair_plot_every = [350]*len(pair_number_iterations)

pair_recipe = {
    "algorithm_list": ["HAPRE", "ER", "HAPRE", "ER", "gradient_descent", "gradient_descent"],
    "number_iterations": pair_number_iterations,
    "helicity": ["pos", "pos", "neg", "neg", "pos", "neg"],
    "beta_zero": [0.5, 0.5, 0.5, 0.5, 1.5e6, 1000.],
    "beta_mode": ["arctan", "const", "arctan", "const", "const", "const"],
    "alpha_zero": [0., 0., 0., 0., 0.1, 0.1],
    "alpha_mode": ["arctan", "const", "arctan", "const", "const", "const"],
    "RL_its": [0, 0, 0, 0,0,0],
    "RL_freqs": [1e9, 1e9, 1e9, 1e9, 1e9, 1e9],
    "TV_freqs": [1e9, 1e9, 1e9, 1e9, 1e9,1e9],
    "plot_every": pair_plot_every,
    "average_img": pair_average_img,
    "Fourier_last": [True, True, True, True, True, True],
    "hologram_intensity_cutoff_vmin": -1,
    "Startimage": [None, "pos", None, "neg", "pos", "neg"],
    "Startgamma": [None, None, None, None, None,None],
    "output": [False, True, False, True, True,True],
}

same_energy_pair_results = {}
for energy_index in np.asarray(PAIR_RETRIEVAL_INDICES, dtype=int)[2:3]:
    pair_mask = mask_pixel_energy[energy_index] if USE_MASK_PIXEL else np.zeros_like(cr_prepared[energy_index])
    results = pr.phase_retrieval_algorithm(
        cr_prepared[energy_index],
        cl_prepared[energy_index],
        pair_mask ,
        supptemp,
        phase_retrieval_recipe=pair_recipe,
    )
    same_energy_pair_results[float(energies_eV[energy_index])] = results
    print(f"retrieved CR/CL pair at {energies_eV[energy_index]:.1f} eV")

print(f"same-energy pair reconstructions: {len(same_energy_pair_results)}")


## Same-energy pair diagnostics

Choose one reconstructed energy and inspect the recipe errors, selected Fourier-domain fields, and their centered real-space Fourier transforms.


In [ ]:
# Choose which same-energy CR/CL pair to inspect.
# Set PAIR_DIAGNOSTIC_ENERGY to an energy in eV, or set
# PAIR_DIAGNOSTIC_ENERGY_INDEX to an index into energies_eV.
PAIR_DIAGNOSTIC_ENERGY = None
PAIR_DIAGNOSTIC_ENERGY_INDEX = None

if not same_energy_pair_results:
    raise RuntimeError("Run the same-energy pair reconstruction cell first.")

available_pair_energies = np.asarray(sorted(same_energy_pair_results), dtype=float)

if PAIR_DIAGNOSTIC_ENERGY_INDEX is not None:
    pair_diagnostic_energy = float(energies_eV[int(PAIR_DIAGNOSTIC_ENERGY_INDEX)])
elif PAIR_DIAGNOSTIC_ENERGY is not None:
    pair_diagnostic_energy = float(PAIR_DIAGNOSTIC_ENERGY)
else:
    pair_diagnostic_energy = float(available_pair_energies[0])

nearest_energy = float(available_pair_energies[np.argmin(np.abs(available_pair_energies - pair_diagnostic_energy))])
pair_diagnostic_results = same_energy_pair_results[nearest_energy]
pair_diagnostic_errors = pair_diagnostic_results[-1]
pair_diagnostic_energy = nearest_energy
pair_diagnostic_energy_index = int(np.argmin(np.abs(energies_eV - pair_diagnostic_energy)))

print("available reconstructed energies (eV):", np.round(available_pair_energies, 3))
print(f"displaying diagnostics for {pair_diagnostic_energy:.3f} eV "
      f"(energy index {pair_diagnostic_energy_index})")


In [ ]:
fig,ax=plt.subplots()
gradient_step = pair_diagnostic_errors["steps"][4]
ax.plot(gradient_step["support_error"])
ax2=ax.twinx()
ax2.plot(gradient_step["loss"], "--", c="red")

In [ ]:
%matplotlib widget
plt.close("all")

steps = pair_diagnostic_errors["steps"]
step_numbers = np.asarray([step["step"] for step in steps])
step_labels = [f"{step['step']}: {step['helicity']} {step['mode']}" for step in steps]

def _last_finite(values):
    values = np.asarray(values, dtype=float).ravel()
    values = values[np.isfinite(values)]
    return np.nan if values.size == 0 else values[-1]

final_diffraction_error = np.asarray([_last_finite(step.get("error", [])) for step in steps])
final_support_error = np.asarray([_last_finite(step.get("support_error", [])) for step in steps])
final_gradient_loss = np.asarray([_last_finite(step.get("loss", [])) for step in steps])

fig, axes = plt.subplots(2,1, figsize=(6, 6))

axes[0].plot(step_numbers, final_diffraction_error, "o-", label="diffraction")
if np.any(np.isfinite(final_support_error)):
    axes[0].plot(step_numbers, final_support_error, "s-", label="support")
if np.any(np.isfinite(final_gradient_loss)):
    axes[0].plot(step_numbers, final_gradient_loss, "^-", label="gradient loss")
axes[0].set_xticks(step_numbers)
axes[0].set_xticklabels(step_labels, rotation=45, ha="right")
axes[0].set_yscale("symlog", linthresh=1e-12)
axes[0].set_xlabel("recipe step")
axes[0].set_ylabel("final error")
axes[0].set_title("Final error by recipe step")
axes[0].legend()

for step in steps:
    err = np.asarray(step.get("error", []), dtype=float).ravel()
    if err.size:
        axes[1].plot(err, label=f"{step['step']}: {step['helicity']} {step['mode']}")
axes[1].set_yscale("symlog", linthresh=1e-12)
axes[1].set_xlabel("iteration within step")
axes[1].set_ylabel("diffraction error")
axes[1].set_title("Error histories")
axes[1].legend(fontsize=8)

fig.suptitle(f"Same-energy CR/CL pair errors at {pair_diagnostic_energy:.1f} eV")
fig.tight_layout()


In [ ]:
if True:
    %matplotlib widget
    plt.close("all")

    pair_outputs = pair_diagnostic_errors.get("outputs", [])
    if pair_outputs:
        pair_fields = {
            f"step {item['step']} {item['helicity']} {item['mode']}": item["field"]
            for item in pair_outputs
        }
    else:
        pair_fields = {
            "latest pos": pair_diagnostic_results[0],
            "latest neg": pair_diagnostic_results[1],
        }

    pair_fields = {label: field for label, field in pair_fields.items() if field is not None}
    if not pair_fields:
        raise RuntimeError("No reconstructed fields are available for the selected energy.")

    if False:
        fig, axes = plt.subplots(2, len(pair_fields), figsize=(3 * len(pair_fields), 6), squeeze=False)
        for column, (label, field) in enumerate(pair_fields.items()):
            axes[0, column].imshow(np.log10(np.abs(field) + 1e-12), cmap="magma")
            axes[0, column].set_title(label)
            axes[0, column].axis("off")
            axes[1, column].imshow(np.angle(field), cmap="twilight", vmin=-np.pi, vmax=np.pi)
            axes[1, column].axis("off")

        axes[0, 0].set_ylabel("log10 |field|")
        axes[1, 0].set_ylabel("phase")
        fig.suptitle(f"Selected Fourier-domain fields at {pair_diagnostic_energy:.1f} eV")
        fig.tight_layout()


In [ ]:
%matplotlib widget
plt.close("all")

def pair_field_to_real_space(field):
    if hasattr(pr, "fourier_field_to_display_object"):
        return pr.fourier_field_to_display_object(field)
    return np.fft.fftshift(np.fft.fft2(np.fft.fftshift(field)))

pair_objects = {label: pair_field_to_real_space(field) for label, field in pair_fields.items()}

roi_for_pair = roi if "roi" in globals() else np.s_[300:410,300:410]
support_for_pair = supptemp if "supptemp" in globals() and supptemp.shape == next(iter(pair_objects.values())).shape else 1

fig, axes = plt.subplots(3, len(pair_objects), figsize=(3 * len(pair_objects), 9), squeeze=False, sharex=True, sharey=True)
for column, (label, obj) in enumerate(pair_objects.items()):
    display_obj = (support_for_pair * obj)[roi_for_pair]
    axes[0, column].imshow(np.real(display_obj), cmap="RdBu_r")
    axes[0, column].set_title(label)
    axes[1, column].imshow(np.imag(display_obj), cmap="twilight")
    axes[2, column].imshow(np.abs(display_obj), cmap="magma")
    for row in range(3):
        axes[row, column].axis("off")

axes[0, 0].set_ylabel("real")
axes[1, 0].set_ylabel("imag")
axes[2, 0].set_ylabel("abs")
fig.suptitle(f"Real-space reconstructions at {pair_diagnostic_energy:.1f} eV")
fig.tight_layout()


## Joint phase retrieval

`QUICK_RUN=True` is a short end-to-end check. Set it to `False` for the longer starting recipe, then tune the iteration counts based on convergence. The physical model is

$$L(E,p)=C+q_c(E)+p\,q_m(E)m_z,$$

with one shared state and beam. No ground-truth arrays are passed to the reconstruction.

In [ ]:
QUICK_RUN = False

if QUICK_RUN:
    inner_Nit = [5, 1]
    outer_iterations = 5
    warmup_Nit = [20, 5]
    physical_iterations = 3
else:
    inner_Nit = [50]
    outer_iterations = 5
    warmup_Nit = [300, 50]
    physical_iterations = 2

recipe = {
    "projection_model": "physical_factorized",#"physical_factorized",
    "inner_mode": ["ER"],
    "inner_Nit": inner_Nit,
    "outer_iterations": outer_iterations,
    "warmup_mode": ["HAPRE", "ER"],
    "warmup_Nit": warmup_Nit,
    "beta_zero": 0.5,
    "beta_mode": "arctan",
    "alpha_zero": 0.,
    "TV_freq": 1e9,
    "alpha_mode": "linear_to_0",
    "average_img": 1,
    "plot_every": 20,
    "shuffle_observations": True,
    "random_seed": 7,
    "projection_every": 2,
    "projection_start": None,
    "projection_relaxation": 1.0,
    "physical_iterations": physical_iterations,
    "energy_values": energies_eV,
    "charge_spectral_constraint": "known_beta",
    "magnetic_spectral_constraint": "known_beta",
    "final_fourier_constraint": False,
    "zero_magnetization_outside_support": False,
    "projection_constraints_inside_support_only": False,
    "energy_values": energies_eV,
    "thickness": 1.,
    "known_charge_beta_spectrum": np.imag(np.sum(effective_refractive_indices_m[:, -2:, 0], axis=1)),
    "known_charge_delta_spectrum": np.real(np.sum(effective_refractive_indices_m[:, -2:, 0], axis=1)),
    "known_magnetic_beta_spectrum": np.imag(np.sum(effective_refractive_indices_m[:, -2:, 1], axis=1)),
    "known_magnetic_delta_spectrum": np.real(np.sum(effective_refractive_indices_m[:, -2:, 1], axis=1)),
}
fields, fieldswarmup,components, bsmasks, errors = pr.universal_phase_retrieval_algorithm(
    holograms,
    retrieval_mask_pixel*0,
    supptemp,
    state_labels=state_labels,
    energy_labels=energy_labels,
    polarization_coefficients=polarizations,
    illumination_labels=illumination_labels,
    saturated_states=None,
    universal_recipe=recipe,
)

try:
    print(f"fit residual RMS: {components['fit_residual_rms']:.4g}")
    print("magnetic scale anchored:", components["magnetic_scale_anchored"])
    print("fully identifiable:", components["identifiable"])
    print(f"runtime: {errors['runtime_seconds']:.1f} s")
except: pass

With one unknown state, the product $q_m(E)m_z$ is identifiable but its factors have a scale/sign ambiguity. The library clips the recovered reduced magnetization to `[-1, 1]`. A known saturated state or an absolutely calibrated magnetic spectrum is required to anchor the physical scale.

In [ ]:
import scipy
label, N=scipy.ndimage.label(supptemp)
counts = np.bincount(label[label!=0])
most_frequent_value = np.argmax(counts)
y0,x0=scipy.ndimage.center_of_mass(label, most_frequent_value)
print(most_frequent_value,y0,x0)
y0,x0=int(y0),int(x0)
r=int(np.sqrt(np.sum(label==most_frequent_value)/np.pi)*1.)
roi=np.s_[np.clip(y0-r,0,None):np.clip(y0+r,0,supptemp.shape[0]-1),np.clip(x0-r,0,None):np.clip(x0+r,0,supptemp.shape[0]-1)]

print(roi)

In [ ]:
%matplotlib widget
plt.close("all")
interactive.cimshow(np.abs(supptemp)[roi])

In [ ]:
magnetization = components["magnetization_by_state"]["fixed_state"]
charge_response = np.asarray(components["charge_response"])
magnetic_response = np.asarray(components["magnetic_response"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
image = axes[0].imshow(magnetization[roi], cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_title("Recovered magnetic state $m_z$")
axes[0].axis("off")
fig.colorbar(image, ax=axes[0], fraction=0.046)

axes[1].plot(energies_eV, np.abs(charge_response), "o-", label="abs")
ax2=axes[1].twinx()
ax2.plot(energies_eV, 2*np.pi*((np.angle(charge_response)/(2*np.pi) +6)%2), "o-", label="phase", color="orange")
axes[1].set_title("Recovered charge response")
axes[1].set_xlabel("Energy (eV)")
axes[1].legend()

axes[2].plot(energies_eV, np.abs(magnetic_response), "o-", label="abs")
ax3=axes[2].twinx()
ax3.plot(energies_eV, np.angle(magnetic_response), "o-", label="phase", color="orange")
axes[2].set_title("Recovered magnetic response")
axes[2].set_xlabel("Energy (eV)")
axes[2].legend()
plt.tight_layout()

## Validate against the simulated XMCD state

The complex `xmcd_logs` obey `log(CR/CL) = 2 q_m(E)m_z`. Because stretching reciprocal-space data by $E/E_{\min}$ contracts the corresponding real-space coordinates by $E_{\min}/E$, each validation map is normalized by that inverse factor before the rank-one decomposition. This validation data was not used by phase retrieval.

In [ ]:
true_xmcd_logs = []
for chunk_file in chunk_files:
    with h5py.File(chunk_file, "r") as handle:
        true_xmcd_logs.append(np.squeeze(np.asarray(handle["xmcd_log/data"])))
true_xmcd_logs = np.stack(true_xmcd_logs)

true_xmcd_logs = np.stack([
    centered_rescale(log_map, 1.0 / scale, output_shape=(ny, nx), order=1, rescale=rescale)
    for log_map, scale in zip(true_xmcd_logs, scale_factors)
])
if crop_shape is not None:
    true_xmcd_logs = np.stack([
        centered_resize(log_map, crop_shape, order=1)
        for log_map in true_xmcd_logs
    ])
if bin_factor > 1:
    true_xmcd_logs = center_crop_stack(true_xmcd_logs, holograms.shape[-2:])

# The first spatial right-singular vector is the common state times an
# arbitrary complex phase. Rotate it to be maximally real, then normalize.
truth_ny, truth_nx = true_xmcd_logs.shape[-2:]
_, _, vh = np.linalg.svd(true_xmcd_logs.reshape(n_energy, -1), full_matrices=False)
truth_complex = vh[0].reshape(truth_ny, truth_nx)
truth_phase = 0.5 * np.angle(np.sum(truth_complex ** 2))
truth_state = np.real(truth_complex * np.exp(-1j * truth_phase))
truth_state /= np.max(np.abs(truth_state))

# The component magnetization is in the log-object frame. The support template
# passed to phase retrieval is in the support-mask frame, so shift it here.
inside = np.fft.fftshift(supptemp.astype(bool))
if np.sum(magnetization[inside] * truth_state[inside]) < 0:
    truth_state *= -1

correlation = np.corrcoef(magnetization[inside], truth_state[inside])[0, 1]
difference = magnetization - truth_state

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for axis, image_data, title in zip(
    axes,
    (magnetization, truth_state, difference),
    ("Phase-retrieval state", "Simulated XMCD state", "Difference"),
):
    image = axis.imshow(image_data, cmap="RdBu_r", vmin=-1, vmax=1)
    axis.set_title(title)
    axis.axis("off")
    fig.colorbar(image, ax=axis, fraction=0.046)
plt.suptitle(f"State correlation inside support: {correlation:.4f}")
plt.tight_layout()

print(f"state correlation inside support: {correlation:.6f}")


In [ ]:
interactive.cimshow(np.abs(np.fft.fftshift(phase)))

In [ ]:
%matplotlib widget
# Inspect reconstructed real-space exit-wave phases at representative energies.
object_logs = pr.fourier_field_to_object_log(fields)
exit_waves = np.exp(object_logs)

fig, axes = plt.subplots(2, len(show_indices), figsize=(2*len(show_indices), 4), squeeze=False)
for column, energy_index in enumerate(show_indices):
    for row, (offset, label) in enumerate(((0, "CR"), (1, "CL"))):
        phase = np.abs(exit_waves[2 * energy_index + offset])
        axes[row, column].imshow((supptemp*np.fft.fftshift(phase))[roi], cmap="twilight", vmin=-np.pi, vmax=np.pi)
        axes[row, column].set_title(f"{label}, {energies_eV[energy_index]:.1f} eV")
        #axes[row, column].axis("off")
plt.suptitle("Reconstructed exit-wave phase")
plt.tight_layout()

In [ ]:
%matplotlib widget
# Inspect reconstructed real-space exit-wave phases at representative energies.
object_logs = pr.fourier_field_to_object_log(fieldswarmup)
exit_waves = np.exp(object_logs)

fig, axes = plt.subplots(2, len(show_indices), figsize=(2*len(show_indices), 4), squeeze=False)
for column, energy_index in enumerate(show_indices):
    for row, (offset, label) in enumerate(((0, "CR"), (1, "CL"))):
        phase = np.imag(exit_waves[2 * energy_index + offset]-offset*exit_waves[2 * energy_index + 0])
        axes[row, column].imshow((np.fft.fftshift(phase)*supptemp)[roi])
        axes[row, column].set_title(f"{label}, {energies_eV[energy_index]:.1f} eV")
        axes[row, column].axis("off")
plt.suptitle("Reconstructed exit-wave phase")
plt.tight_layout()

%matplotlib widget
# Inspect reconstructed real-space exit-wave phases at representative energies.
object_logs = pr.fourier_field_to_object_log(fields)
exit_waves = np.exp(object_logs)

fig, axes = plt.subplots(2, len(show_indices), figsize=(2*len(show_indices), 4), squeeze=False)
for column, energy_index in enumerate(show_indices):
    for row, (offset, label) in enumerate(((0, "CR"), (1, "CL"))):
        phase = np.imag(exit_waves[2 * energy_index + offset]-offset*exit_waves[2 * energy_index + 0])
        axes[row, column].imshow((np.fft.fftshift(phase)*supptemp)[roi])
        axes[row, column].set_title(f"{label}, {energies_eV[energy_index]:.1f} eV")
        axes[row, column].axis("off")
plt.suptitle("Reconstructed exit-wave phase")
plt.tight_layout()

In [ ]:
fig,ax=plt.subplots(2,2)
ax[0,0].plot(np.abs(components["charge_response"]))
ax[0,1].plot(np.abs(components["magnetic_response"]))
ax[1,0].plot(2*np.pi*(np.angle(components["charge_response"]/(2*np.pi) +6)%1))
ax[1,0].plot(2*np.pi*(np.angle(components["charge_response"]/(2*np.pi) +6)%1)-2*np.pi)
ax[1,1].plot(2*np.pi*((np.angle(components["magnetic_response"]/(2*np.pi) +6)%1)))
ax[1,1].plot(2*np.pi*((np.angle(components["magnetic_response"]/(2*np.pi) +6)%1))-2*np.pi)

In [ ]:
plt.close("all")
%matplotlib inline
fig,ax=plt.subplots()
temp=(np.real(np.fft.fftshift(components["magnetization"][0])))
mi,ma=np.percentile(temp,  (2,98))


In [ ]:
%matplotlib widget
temp=(np.real(np.fft.fftshift(components["magnetization"][0])))
interactive.cimshow(temp[roi], vmin=mi, vmax=ma)

In [ ]:
list(components.keys())

In [ ]:
%matplotlib widget
i=8
print(list(components.keys())[i])
temp=(np.real(np.fft.fftshift(components[list(components.keys())[i]][0])))
interactive.cimshow(temp[roi], vmin=mi, vmax=ma)

In [ ]:
list(components.keys()[i]